# SMX 树木素材：架构、图层字节解析与完整重建

2026-10-03。目标目录全部 110 个 SMX，共 20,229,293 字节、1,914 帧、3,802 层：主图 1,914、阴影 1,879、空轮廓 9。逐文件解析、字节账本与精确重建均通过，64 种 SHA-256 内容，重复文件仍逐个验证。

树桩由作者故意留空以在游戏中屏蔽，此处仅保留并检查其占位结构，不分析视觉内容。

本 notebook 自包含，前期脚本在 tmp 探索后才整理。素材直接引用 reference，未复制 SMX、调色板、图片或原始像素。运行时预览的输出不会保留在交付文件。提交重新运行的 notebook 前，应清除素材输出。

In [1]:
from pathlib import Path
from collections import Counter
import struct
import hashlib
import json
from IPython.display import display, Markdown

# Change this constant to select another local mod.
MOD_ROOT = Path('reference/22014_Anne_HK - Identical Pine Trees with Grid Shadow')
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'reference/reference.md').is_file())
if not MOD_ROOT.is_absolute():
    MOD_ROOT = PROJECT_ROOT / MOD_ROOT
GRAPHICS = MOD_ROOT / 'resources/_common/drs/graphics'
PALETTE_ROOT = PROJECT_ROOT / 'reference/SMX-Workshop/palettes'
assert GRAPHICS.is_dir()


## 依据与验证范围

从 [reference/reference.md](../../reference/reference.md) 开始。主要字节依据是本地 [SpriteIO.java](../../reference/SMX-Workshop/src/com/imwg/smxworkshop/sprite/SpriteIO.java)，checkout commit ea461434f67fd8a421bedc4e383eefc5d1b5ba24：loadFromSMX（98 起）、CompiledSMXFrame（1644 起）、saveSMXSprite（2071 起）。字段与算法依据本地代码以及全部目标文件；本 notebook 为独立 Python 实现。调色板配置来自 palettes/palettes.conf。

交叉参考：[openage SMX 说明](https://github.com/SFTtech/openage/blob/master/doc/media/smx-files.md)、[SMX-Workshop](https://github.com/ImWG/SMX-Workshop)。openage 的 flags 位表与示例矛盾，这里采用 Workshop 实际掩码；其 4plus1 的 section 位序也与本地代码不同，以下采用实际读写验证通过的顺序。网页于 2026-10-03 查阅。

已查看本地 sld-extractor/README.md 及文件列表：这是 SLD 工具，不能拿它的块布局解析 SMX。SMX 与 SLD 必须分开。

此目录未出现 8to5、玩家色和非空轮廓，另用合成数据验证，不能声称这些分支已经真实游戏素材验证。unknown 字段语义未知，但完整保留。游戏兼容性与精确阴影混合需另做游戏内验证。

## 顺序架构与字段

所有多字节数 little-endian。没有帧或层的绝对偏移表，依帧数、flags、层高、流长度顺序读至 EOF。只存存在的层，顺序固定 main → shadow → outline。不存在与存在但尺寸为零不同。

~~~text
file header (32)
  frame header (6)
    main header (16) → edges (4H) → C,P (8) → commands(C) → pixels(P)
    shadow header (16) → edges (4H) → C (4) → commands + values(C)
    outline header (16) → edges (4H) → C (4) → commands(C)
  next frame ... → EOF
~~~

| File offset | 类型 / 字节 | 意义 |
|---|---|---|
| 0 | bytes / 4 | SMPX magic |
| 4 | uint16 / 2 | version，实测全 2 |
| 6 | uint16 / 2 | frame count |
| 8 | uint32 / 4 | compressed，通常文件长度减 32 |
| 12 | uint32 / 4 | SMP/解压大小元数据，不用于读游标 |
| 16 | bytes / 16 | memo，原样保留，不能假定全零 |

| Frame offset | 类型 | 意义 |
|---|---|---|
| 0 | uint8 | flags: 0x01 main、0x02 shadow、0x04 outline、0x08 8to5（否则 4plus1） |
| 1 | uint8 | palette ID，外部色表编号，不是像素索引 |
| 2 | uint32 | smp_size，保留，不用于定位下一帧 |

0x10 接收其他对象阴影的含义来自 openage，样本未出现，无法实测。解析器保留该位，拒绝更高未知位。实测 flags 仅 1、3、7。

| Layer offset | 类型 | 意义 |
|---|---|---|
| 0,2 | uint16 × 2 | width,height |
| 4,6 | int16 × 2 | hotspot x,y，各层锚点 |
| 8 | uint32 | payload 长度声明，不含自身 16-byte header |
| 12 | uint32 | unknown，实测全 0，保留 |
| 16 | height × (uint16,uint16) | left,right 行首尾透明像素数 |

行边界 (65535,65535) 表示全透明行，**不消费任何命令，包括 EOL**。普通行从 x=left 开始，在 x=width−right 结束，内部透明洞由 skip 表达。edges 是边界表，不是轮廓图层，也不是流偏移表。

main payload=4H+8+C+P；shadow/outline payload=4H+4+C。实际解析依 H、C、P 前进，声明长度仅用于诊断，不能依据陈旧字段跳转。

## 各层命令和像素编码

命令 b 的 op=b&3、n=(b>>2)+1，n 在 1–64。op=3 是 EOL，无像素数量，其高六位原样保留。

| op | Main | Shadow | Outline |
|---|---|---|---|
| 0 | skip n 透明像素 | 同左 | 同左 |
| 1 | 独立像素流取 n 普通色 | 命令后 n 个 uint8 值 | 标记 n 个轮廓像素，无附带值 |
| 2 | 独立像素流取 n 玩家色 | Workshop 同 op=1 | Workshop 同 op=1 |
| 3 | EOL | EOL | EOL |

主图像素跨命令、跨行连续打包；skip、EOL 均不消耗槽，**不能每行重置槽索引**。普通像素与玩家色分别保留命令类型，透明用 None，不能用索引 0 替代。

**4plus1（全部真实样本）：**每块 [a,b,c,d,e] 五字节，四个 10-bit palette 索引。第 i 槽（0–3）为 low[i] | (((e>>(2*i))&3)<<8)。e bits 0–1 属于第一个，2–3 属于第二个，以此递增。此方向以 Workshop 代码与实际往返为依据。最后不足四个像素时仍保存整块，未使用槽也须保留；本样本尾槽均为零。

**8to5（合成样本）：**五字节保存两个 10-bit 索引及两个 10-bit smudge/damage 值：

~~~python
p0 = a | ((b & 3) << 8)
p1 = (b >> 2) | ((c & 15) << 6)
s0 = (c >> 4) | ((d & 63) << 4)
s1 = (d >> 6) | (e << 2)
~~~

smudge 是主图像素的附加数据，不是第四个 SMX 独立层。保留全部 10 位，不采用 Workshop 预览时的灰度缩放，否则不能精确复现。

阴影无上述压缩；draw 后立即跟 n 字节强度。值 0 也表示存在的像素，与 None 不同。样本强度范围 0–186。运行时预览以值/255 作为黑色 alpha，准确游戏混合语义未验证。

轮廓 draw 只标记掩码，不读取值。此目录的 9 个轮廓层均是 0×0 占位；非空分支通过合成样本验证。

## 完整解析与精确重建

rows 存像素语义；events 存原命令分段、字节、行坐标、流内起点；slots 存全部使用与未使用的主图槽。spans 是无缝半开区间字节账本 [start,end)，层与帧另存 offset/end。阴影流包括内联值，events 标记的是命令位置。

rebuild 从字段、事件和已解码像素重建，不直接返回原文件或复制原像素流。保留 memo、未知字段、声明大小、空行、原命令字节、分段与多余槽。assert 检查边界、行宽、命令完整消费、像素槽、全文件连续覆盖与 EOF。运行时不要使用 Python -O，它会关闭断言。

In [2]:
def unpack_pixels(data: bytes, enhanced: bool) -> list[tuple[int, int]]:
    assert len(data) % 5 == 0
    result = []
    for p in range(0, len(data), 5):
        a, b, c, d, e = data[p:p+5]
        if enhanced:
            result.extend([(a | ((b & 3) << 8), (c >> 4) | ((d & 63) << 4)),
                           ((b >> 2) | ((c & 15) << 6), (d >> 6) | (e << 2))])
        else:
            result.extend((v | (((e >> (2*i)) & 3) << 8), 0)
                          for i, v in enumerate((a,b,c,d)))
    return result

def pack_pixels(slots: list[tuple[int, int]], enhanced: bool) -> bytes:
    size = 2 if enhanced else 4
    assert len(slots) % size == 0
    out = bytearray()
    for j in range(0, len(slots), size):
        group = slots[j:j+size]
        if enhanced:
            (p,s),(q,t) = group
            out.extend([p & 255, (p >> 8) | ((q & 63) << 2),
                        (q >> 6) | ((s & 15) << 4),
                        (s >> 4) | ((t & 3) << 6), t >> 2])
        else:
            out.extend(p & 255 for p,_ in group)
            out.append(sum((p >> 8) << (2*i) for i,(p,_) in enumerate(group)))
    return bytes(out)

def decode_layer(layer: dict, enhanced: bool) -> None:
    kind = layer['kind']
    stream = layer['commands']
    slots = unpack_pixels(layer['data'], enhanced) if kind == 'main' else []
    rows, events = [], []
    pos = used = 0
    counts = Counter()
    for y,(left,right) in enumerate(layer['edges']):
        row = [None] * layer['width']
        if left == 65535 or right == 65535:
            assert left == right == 65535
            rows.append(row)
            continue
        assert left + right <= layer['width']
        x = left
        while True:
            assert pos < len(stream), (kind,y,pos)
            start = pos
            cmd = stream[pos]; pos += 1
            op, n = cmd & 3, (cmd >> 2) + 1
            counts[op] += 1
            event = {'row': y, 'x': x, 'offset': start, 'byte': cmd, 'op': op, 'count': 0 if op == 3 else n}
            if op == 3:
                assert x == layer['width'] - right, (kind,y,x,layer['width']-right)
                events.append(event)
                break
            assert x + n <= layer['width'] - right
            if op:
                if kind == 'main':
                    assert used + n <= len(slots)
                    for i in range(n):
                        row[x+i] = (op, *slots[used+i])
                    used += n
                elif kind == 'shadow':
                    assert pos + n <= len(stream)
                    for i,v in enumerate(stream[pos:pos+n]):
                        row[x+i] = (op,v)
                    pos += n
                else:
                    for i in range(n):
                        row[x+i] = (op,)
            events.append(event)
            x += n
        rows.append(row)
    assert pos == len(stream), (kind,pos,len(stream))
    if kind == 'main':
        group = 2 if enhanced else 4
        assert len(slots) >= ((used + group - 1) // group) * group
        assert pack_pixels(slots, enhanced) == layer['data']
    layer.update(rows=rows, events=events, slots=slots, used_slots=used, op_counts=dict(counts))

def parse_smx(raw: bytes) -> dict:
    pos = 0
    spans = []
    def read(fmt: str, name: str) -> tuple:
        nonlocal pos
        st = struct.Struct('<'+fmt)
        assert pos + st.size <= len(raw), name
        value = st.unpack_from(raw,pos)
        spans.append((pos,pos+st.size,name)); pos += st.size
        return value
    def take(n: int, name: str) -> bytes:
        nonlocal pos
        assert pos+n <= len(raw), name
        value=raw[pos:pos+n]
        spans.append((pos,pos+n,name)); pos += n
        return value
    signature,version,count,compressed,uncompressed,comment = read('4sHHII16s','file.header')
    assert signature == b'SMPX'
    frames=[]
    for i in range(count):
        start=pos
        flags,palette,smp_size=read('BBI',f'frame.{i}.header')
        assert flags & ~31 == 0, flags
        layers=[]
        for bit,kind in enumerate(('main','shadow','outline')):
            if not flags & (1 << bit): continue
            name=f'frame.{i}.{kind}'
            offset=pos
            w,h,hx,hy,length,unknown=read('HHhhII',name+'.header')
            edges=[read('HH',name+f'.edge.{y}') for y in range(h)]
            cl,=read('I',name+'.command_length')
            dl,=read('I',name+'.pixel_length') if kind=='main' else (0,)
            commands=take(cl,name+'.commands')
            data=take(dl,name+'.pixels') if kind=='main' else b''
            layer=dict(kind=kind,offset=offset,end=pos,width=w,height=h,hotspot=(hx,hy),
                       length=length,unknown=unknown,edges=edges,commands=commands,data=data)
            decode_layer(layer,bool(flags & 8))
            layers.append(layer)
        frames.append(dict(offset=start,end=pos,flags=flags,palette=palette,smp_size=smp_size,layers=layers))
    assert pos == len(raw), (pos,len(raw))
    cursor=0
    for a,b,_ in spans:
        assert a == cursor
        cursor=b
    assert cursor == len(raw)
    return dict(version=version,compressed=compressed,uncompressed=uncompressed,comment=comment,frames=frames,spans=spans)

def rebuild(model: dict) -> bytes:
    # Rebuild fields, commands, shadow values and every packed pixel slot.
    body=bytearray()
    for f in model['frames']:
        body.extend(struct.pack('<BBI',f['flags'],f['palette'],f['smp_size']))
        for l in f['layers']:
            stream=bytearray()
            live=[]
            for event in l['events']:
                stream.append(event['byte'])
                op,n=event['op'],event['count']
                if op in (1,2):
                    pixels=l['rows'][event['row']][event['x']:event['x']+n]
                    assert all(p[0] == op for p in pixels)
                    if l['kind']=='main': live.extend((p[1],p[2]) for p in pixels)
                    elif l['kind']=='shadow': stream.extend(p[1] for p in pixels)
            data=pack_pixels(live+l['slots'][l['used_slots']:],bool(f['flags'] & 8)) if l['kind']=='main' else b''
            edges=b''.join(struct.pack('<HH',*e) for e in l['edges'])
            sizes=struct.pack('<II',len(stream),len(data)) if l['kind']=='main' else struct.pack('<I',len(stream))
            payload=edges+sizes+stream+data
            body.extend(struct.pack('<HHhhII',l['width'],l['height'],*l['hotspot'],l['length'],l['unknown']))
            body.extend(payload)
    return struct.pack('<4sHHII16s',b'SMPX',model['version'],len(model['frames']),model['compressed'],model['uncompressed'],model['comment'])+body

def audit(directory: Path) -> tuple[list[dict], dict]:
    files=sorted(directory.iterdir())
    assert files and all(p.is_file() and p.suffix.lower()=='.smx' for p in files)
    records=[]
    totals=Counter()
    values={k:set() for k in ('flags','palette','version','unknown','comment','shadow','padding','dimensions','hotspots')}
    hashes=Counter()
    for path in files:
        raw=path.read_bytes(); m=parse_smx(raw)
        rebuilt=rebuild(m)
        assert rebuilt == raw, path.name
        hashes[hashlib.sha256(raw).hexdigest()]+=1
        layers=[l for f in m['frames'] for l in f['layers']]
        record=dict(file=path.name,bytes=len(raw),sha256=hashlib.sha256(raw).hexdigest(),frames=len(m['frames']),
                    layers=len(layers),flags=sorted({f['flags'] for f in m['frames']}),
                    palettes=sorted({f['palette'] for f in m['frames']}),roundtrip=True,
                    spans=len(m['spans']),
                    compressed_delta=m['compressed']-(len(raw)-32),
                    smp_delta=m['uncompressed']-(4*len(m['frames'])+sum(f['smp_size'] for f in m['frames'])),
                    stale_layers=[dict(frame=i,kind=l['kind'],declared=l['length'],actual=l['end']-l['offset']-16) for i,f in enumerate(m['frames']) for l in f['layers'] if l['length'] != l['end']-l['offset']-16],
                    extra_slot_layers=[dict(frame=i,kind=l['kind'],used=l['used_slots'],stored=len(l['slots'])) for i,f in enumerate(m['frames']) for l in f['layers'] if l['kind']=='main' and len(l['slots']) > ((l['used_slots']+3)//4)*4])
        records.append(record)
        totals.update(files=1,bytes=len(raw),frames=len(m['frames']),layers=len(layers))
        values['version'].add(m['version']);values['comment'].add(m['comment'].hex())
        for f in m['frames']:
            values['flags'].add(f['flags']); values['palette'].add(f['palette'])
            for l in f['layers']:
                totals[l['kind']]+=1
                totals['empty_rows']+=sum(a==65535 for a,b in l['edges'])
                totals['player_commands']+=l['op_counts'].get(2,0) if l['kind']=='main' else 0
                values['unknown'].add(l['unknown']);values['dimensions'].add((l['kind'],l['width'],l['height']))
                values['hotspots'].add((l['kind'],*l['hotspot']))
                values['padding'].update(l['slots'][l['used_slots']:])
                if l['kind']=='shadow':values['shadow'].update(p[1] for row in l['rows'] for p in row if p)
    return records,dict(totals=dict(totals),values={k:sorted(v) for k,v in values.items()},unique_files=len(hashes),hash_groups=sorted(hashes.values()))



## 全目录验证和逐文件清单

枚举所有直接子项，并断言全部为 SMX 文件；避免 glob 悄悄漏掉其他内容。每个文件单独解析与重建，并直接字节比较，SHA-256 只是清单指纹。

In [3]:
records, summary = audit(GRAPHICS)
print(json.dumps({'totals': summary['totals'], 'unique_files': summary['unique_files'], 'flags': summary['values']['flags'], 'palettes': summary['values']['palette']}, indent=2))
lines = ['| File | Bytes | Frames | Layers | SHA-256 | Exact rebuild |', '|---|---:|---:|---:|---|---|']
lines += [f"| {r['file']} | {r['bytes']} | {r['frames']} | {r['layers']} | {r['sha256']} | {r['roundtrip']} |" for r in records]
display(Markdown('\n'.join(lines)))


{
  "totals": {
    "files": 110,
    "bytes": 20229293,
    "frames": 1914,
    "layers": 3802,
    "main": 1914,
    "empty_rows": 26788,
    "player_commands": 0,
    "shadow": 1879,
    "outline": 9
  },
  "unique_files": 64,
  "flags": [
    1,
    3,
    7
  ],
  "palettes": [
    20,
    28,
    30
  ]
}


| File | Bytes | Frames | Layers | SHA-256 | Exact rebuild |
|---|---:|---:|---:|---|---|
| n_tree_acacia_x1.smx | 142292 | 30 | 60 | d7bba2a371a20d0219b43b802deae15e984fa5bf5cb7071c8d1d46ae437ff13d | True |
| n_tree_acacia_x2.smx | 507122 | 30 | 60 | 5c94e8f370690af9e0b982f5bf1f61e6236b4a80b195267bdba81a0d019feacd | True |
| n_tree_asian_maple_autumn_x1.smx | 85388 | 18 | 36 | d6e81060bad9c4de71849e9cb685c1fbee3e4cdda37b330192ace43754669047 | True |
| n_tree_asian_maple_autumn_x2.smx | 304286 | 18 | 36 | c3d6ac4b2d970513dc8e52576bc8800f2b82c48e2646a2163e1789ce4d3a729f | True |
| n_tree_asian_maple_green_x1.smx | 42710 | 9 | 18 | f318cb9f4d5728760718956040e8d15aa54527e90f9d096414b329d5a5e774b4 | True |
| n_tree_asian_maple_green_x2.smx | 152159 | 9 | 18 | a214b2b7dcf1ce18fce301be0e1666fa1bb9abfb8e06afba27e2001e13cce6ef | True |
| n_tree_asian_pine_x1.smx | 19000 | 4 | 8 | 8237a903fc7a3fedf36ca56caa1ff2d62b9908fab4443866aa7b5e8605d5e1ad | True |
| n_tree_asian_pine_x2.smx | 67644 | 4 | 8 | 64813674cc15614861a9c8c1e7948c003dd6cda8261dd28b6263e95d2d28fd8d | True |
| n_tree_autumn_oak_x1.smx | 199196 | 42 | 84 | 1921bdf9c914a5ec52d79309336d28c57d344710f1f770e0d73b3b9c86ad9416 | True |
| n_tree_autumn_oak_x2.smx | 709958 | 42 | 84 | c8ac2b83871022ae6cdd90856169b66a3f1c78a809144c59862592ec08f2baa7 | True |
| n_tree_bamboo_x1.smx | 56936 | 12 | 24 | ffe08a5f545e276a8b4d5941b887d07d321b4dbd07d07c5e3b4ea6deb349a907 | True |
| n_tree_bamboo_x2.smx | 202868 | 12 | 24 | 1b947656bcd7eb755e198fc8bb452fabd58c57b75ee52017dcf74ff48b7dd442 | True |
| n_tree_baobab_x1.smx | 251267 | 45 | 90 | 8e91ccfac4256d0205c8f13fc0b141396182908c4f87ef180c417081c9a4e58a | True |
| n_tree_baobab_x2.smx | 909662 | 45 | 90 | a9f88427eb8c33c582ed35de0962cc49f650433d63af81e65ba550180d3d1c0b | True |
| n_tree_birch_x1.smx | 199742 | 42 | 84 | 9bda6ee7a5826ff8706a7b3c90bec2e15e555480e19056dfbfc1aa38c4953b62 | True |
| n_tree_birch_x2.smx | 709958 | 42 | 84 | b29a21cbe5afb3e3647cda33930ac95ee304a6b4ac69ec1627463ac000d7fe03 | True |
| n_tree_brazilwood_x1.smx | 66420 | 14 | 28 | cdeeb03d356c48f612364658d880b16901fad8c73c89bcf4811ff7021b61f69a | True |
| n_tree_brazilwood_x2.smx | 236674 | 14 | 28 | e18bc8b3c9c203df966f3a84392ddb0d5f209f07354845b4f43201e2a16d2786 | True |
| n_tree_bush_a_x1.smx | 28484 | 6 | 12 | 383f3abb80cd8a6156b89b3493f0130ee8659efa96845ba47c39ccd8a377646d | True |
| n_tree_bush_a_x2.smx | 101450 | 6 | 12 | a47f5775f4395cfecc4cbe0766020c3c5cf0c575a83dce4977faf5275d459bc5 | True |
| n_tree_bush_b_x1.smx | 256100 | 54 | 108 | 7acab2253db39d9908f0c44632313cf035852ab8f3c925c9572851b3a375b7b6 | True |
| n_tree_bush_b_x2.smx | 912794 | 54 | 108 | 441577e624c050b5d081062e00c55989cc14bc1c197eaeeb2c4ca1687da0f0b0 | True |
| n_tree_bush_c_x1.smx | 128066 | 27 | 54 | 9c63102683c6aa761fc0f99e044d4a5fb71dcadcf1e964bd83f89d755d66516f | True |
| n_tree_bush_c_x2.smx | 456413 | 27 | 54 | 759727e54f0defb589e5517d9652db6e8f97291b8702af76760d46c56ce3fa10 | True |
| n_tree_bush_d_x1.smx | 37968 | 8 | 16 | 357da8daf37a76f39eb5eaa5a073eeeb482b8910935af0fe5063af57911c7e7f | True |
| n_tree_bush_d_x2.smx | 135256 | 8 | 16 | d1967e843211374eaf6784a850ef5731e78c25f681d4a1eaca4a3785e3123010 | True |
| n_tree_cypress_x1.smx | 56936 | 12 | 24 | ffe08a5f545e276a8b4d5941b887d07d321b4dbd07d07c5e3b4ea6deb349a907 | True |
| n_tree_cypress_x2.smx | 202868 | 12 | 24 | 1b947656bcd7eb755e198fc8bb452fabd58c57b75ee52017dcf74ff48b7dd442 | True |
| n_tree_dead_x1.smx | 99614 | 21 | 42 | 649c35ab24a218c5e35d08602b03334359bc0c9c69a4c55edab9f475210d236c | True |
| n_tree_dead_x2.smx | 354995 | 21 | 42 | bd25337d10eff3676b065d178cd0a346df724b6f7d4c1288911f7ecbb5143476 | True |
| n_tree_dragon_x1.smx | 170744 | 36 | 72 | e2128b1ae6c3008a790a19b34e8312f1cc8df365685f6802ba863b206c7d6ab3 | True |
| n_tree_dragon_x2.smx | 608540 | 36 | 72 | d8f7b46f74fb277455174d570cd0f678b26d3cec4e2b22a8195b9cfecdae9590 | True |
| n_tree_felled_bamboo_x1.smx | 31116 | 9 | 18 | 752e3aac3eb7ca48bbc4588c0286acdd720dbbed76f85b3ace4414a60b6187a9 | True |
| n_tree_felled_bamboo_x2.smx | 109426 | 9 | 18 | d64441fa725ee9747a4577ab9c65b18458d0c71136e8ad6e5cf69a39af45255c | True |
| n_tree_felled_baobab_x1.smx | 51843 | 15 | 30 | 7cfb683cc64a17a4cb4b68377a38865d978cfab6478438ac27bc35ec3b615c4b | True |
| n_tree_felled_baobab_x2.smx | 182291 | 15 | 30 | 1aa65244bf895685e226fd5292f1bef2ac3e7965a4874ded6057338098e9d958 | True |
| n_tree_felled_generic_x1.smx | 20759 | 6 | 12 | 207d3b24396cf3189c20505f1c059c067f79d90f8ca44a573f4f0d250e160524 | True |
| n_tree_felled_generic_x2.smx | 73294 | 6 | 12 | 4a346de4c413f4d4ab21326a259f4f7fafc58fe563fc19ba196156ab1655c425 | True |
| n_tree_felled_lush_bamboo_x1.smx | 13876 | 4 | 8 | 70a5df2ea06ef23c1681c01fbe013d5e22d05409ddb921b8ef0b5cd6b7787ccb | True |
| n_tree_felled_lush_bamboo_x2.smx | 48572 | 4 | 8 | aa1de638dd9aa334f9b31bdd79d80ae94780118f3812ff057e7ce9378b281014 | True |
| n_tree_green_oak_x1.smx | 128066 | 27 | 54 | eccec33d1891fbc09f2d84e047d30620dbe7f2111cded49884beb1cea86586ea | True |
| n_tree_green_oak_x2.smx | 456413 | 27 | 54 | e898ba3eeabba36e762564711d1308c76048e088d8cfae12484338fe58d3a049 | True |
| n_tree_italian_pine_x1.smx | 113840 | 24 | 48 | 4c248174337b902536e2f94e9aa21afb3af39bee01bce78e54c606461b6062a5 | True |
| n_tree_italian_pine_x2.smx | 405704 | 24 | 48 | 07d193a57aff698be15dacff899f0f9885d4902810928a191b1e4ab1fcc09f7b | True |
| n_tree_jungle_x1.smx | 184970 | 39 | 78 | 774d7f16fac1acc1e85e403acb422982a7f2662341fa650f39b831153a897d76 | True |
| n_tree_jungle_x2.smx | 659249 | 39 | 78 | daf2803c165e9796507087a119bb0d5f01c34cc189c9005adf49d7d924d5e419 | True |
| n_tree_lush_bamboo_x1.smx | 75904 | 16 | 32 | 06b40edab823eb64bb97b72932f8744b66a63bb328d7bf220361a8c32d189744 | True |
| n_tree_lush_bamboo_x2.smx | 270480 | 16 | 32 | 3b69c9bf13115be1fd171d113e55b00e5f1bb32445b2f97ba7d932a88ea42d15 | True |
| n_tree_mangrove_x1.smx | 170744 | 36 | 72 | e2128b1ae6c3008a790a19b34e8312f1cc8df365685f6802ba863b206c7d6ab3 | True |
| n_tree_mangrove_x2.smx | 608540 | 36 | 72 | d8f7b46f74fb277455174d570cd0f678b26d3cec4e2b22a8195b9cfecdae9590 | True |
| n_tree_monkey_puzzle_x1.smx | 37968 | 8 | 16 | 357da8daf37a76f39eb5eaa5a073eeeb482b8910935af0fe5063af57911c7e7f | True |
| n_tree_monkey_puzzle_x2.smx | 135256 | 8 | 16 | d1967e843211374eaf6784a850ef5731e78c25f681d4a1eaca4a3785e3123010 | True |
| n_tree_oak_x1.smx | 199196 | 42 | 84 | 1921bdf9c914a5ec52d79309336d28c57d344710f1f770e0d73b3b9c86ad9416 | True |
| n_tree_oak_x2.smx | 709958 | 42 | 84 | c8ac2b83871022ae6cdd90856169b66a3f1c78a809144c59862592ec08f2baa7 | True |
| n_tree_olive_x1.smx | 113840 | 24 | 48 | 4c248174337b902536e2f94e9aa21afb3af39bee01bce78e54c606461b6062a5 | True |
| n_tree_olive_x2.smx | 405704 | 24 | 48 | 07d193a57aff698be15dacff899f0f9885d4902810928a191b1e4ab1fcc09f7b | True |
| n_tree_palm_x1.smx | 184970 | 39 | 78 | 774d7f16fac1acc1e85e403acb422982a7f2662341fa650f39b831153a897d76 | True |
| n_tree_palm_x2.smx | 659249 | 39 | 78 | daf2803c165e9796507087a119bb0d5f01c34cc189c9005adf49d7d924d5e419 | True |
| n_tree_peach_blossom_x1.smx | 23742 | 5 | 10 | 17fc6d7c888592b623a34d7bd71b31e957afd879c90d336d32b2af7c117f4f1f | True |
| n_tree_peach_blossom_x2.smx | 84547 | 5 | 10 | 1e9759d557076fb279a5d017e10bef760fac14f6c968906a9bcfee6efa0eb94a | True |
| n_tree_pine_x1.smx | 128066 | 27 | 54 | 9c63102683c6aa761fc0f99e044d4a5fb71dcadcf1e964bd83f89d755d66516f | True |
| n_tree_pine_x2.smx | 456413 | 27 | 54 | 759727e54f0defb589e5517d9652db6e8f97291b8702af76760d46c56ce3fa10 | True |
| n_tree_rainforest_x1.smx | 327230 | 69 | 138 | 2ae2c3b1074c8ddd2c53ec76412894a96f7e95acf2bbc45246e735aa443f93a6 | True |
| n_tree_rainforest_x2.smx | 1166339 | 69 | 138 | 62b21d19eb4f75b39e3155dca2c08436e1a2b2aca36ab322e242e44721713186 | True |
| n_tree_reeds_x1.smx | 56936 | 12 | 24 | ffe08a5f545e276a8b4d5941b887d07d321b4dbd07d07c5e3b4ea6deb349a907 | True |
| n_tree_reeds_x2.smx | 202868 | 12 | 24 | 1b947656bcd7eb755e198fc8bb452fabd58c57b75ee52017dcf74ff48b7dd442 | True |
| n_tree_scenario_a_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_a_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_b_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_b_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_c_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_c_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_d_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_d_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_e_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_e_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_f_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_f_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_g_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_g_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_h_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_h_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_i_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_i_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_j_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_j_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_k_x1.smx | 14258 | 3 | 6 | b7c38f6efcae955b81a22b47da4bf50017f5341f80db0ea5ef96025722921a39 | True |
| n_tree_scenario_k_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_scenario_l_x1.smx | 14258 | 3 | 6 | 880936dbcdf6de9e0c2f0fe48bc70a923b00f32056b63c4cf9464d5b091f6e76 | True |
| n_tree_scenario_l_x2.smx | 50741 | 3 | 6 | f966b46fd85967f03114cc58024d4c21dc8b9ffaf5a24118660abdd31e52f359 | True |
| n_tree_snow_autumn_oak_x1.smx | 199196 | 42 | 84 | 1921bdf9c914a5ec52d79309336d28c57d344710f1f770e0d73b3b9c86ad9416 | True |
| n_tree_snow_autumn_oak_x2.smx | 709958 | 42 | 84 | c8ac2b83871022ae6cdd90856169b66a3f1c78a809144c59862592ec08f2baa7 | True |
| n_tree_snow_pine_x1.smx | 128066 | 27 | 54 | 9c63102683c6aa761fc0f99e044d4a5fb71dcadcf1e964bd83f89d755d66516f | True |
| n_tree_snow_pine_x2.smx | 456413 | 27 | 54 | 759727e54f0defb589e5517d9652db6e8f97291b8702af76760d46c56ce3fa10 | True |
| n_tree_spruce_snow_x1.smx | 56936 | 12 | 24 | 350241dd18b567b0b9a3d2c35e265de198bf04d22d7762ecaeecaabe246b5a8f | True |
| n_tree_spruce_snow_x2.smx | 202868 | 12 | 24 | b52da49cb11c4a5dcba48ef4367def1882d9c2f2b26672591fa5c8153db36818 | True |
| n_tree_spruce_x1.smx | 118582 | 25 | 50 | ea7dcb11e49b859cd88e6ba4d3f9987051c5f2788dde32a13a803bce5846bfbe | True |
| n_tree_spruce_x2.smx | 422607 | 25 | 50 | 784f9113edf867f5f45f2c9afce5e09405d38fe2e2da9b9117d0b0825146e4be | True |
| n_tree_stump_bamboo_x1.smx | 1409 | 3 | 9 | abec1db272af799af7230471a8c5845210e878a2fb2d5642f1c29e10b07c4097 | True |
| n_tree_stump_bamboo_x2.smx | 134 | 3 | 3 | 811dcf3939311c305d323352d579740355f6bd0616039c4c481dbbc0ea23e6f7 | True |
| n_tree_stump_baobab_x1.smx | 1409 | 3 | 9 | abec1db272af799af7230471a8c5845210e878a2fb2d5642f1c29e10b07c4097 | True |
| n_tree_stump_baobab_x2.smx | 542 | 15 | 15 | 62e92a3dcbc4a0abb8b59d431edaab0c3f26e05829ad1c2b8ee1aba90e7a31f9 | True |
| n_tree_stump_generic_x1.smx | 1409 | 3 | 9 | abec1db272af799af7230471a8c5845210e878a2fb2d5642f1c29e10b07c4097 | True |
| n_tree_stump_generic_x2.smx | 338 | 9 | 9 | 129b3f093cdd66236340d0b49db1e0a800ebebfc8d013355c962d39eb0864f25 | True |
| n_tree_stump_lush_bamboo_x1.smx | 168 | 4 | 4 | 33571cc41b517f0d1a646dc837847521e31ec4d404d4982fc34176532e47d341 | True |
| n_tree_stump_lush_bamboo_x2.smx | 168 | 4 | 4 | 33571cc41b517f0d1a646dc837847521e31ec4d404d4982fc34176532e47d341 | True |
| n_tree_wax_palm_x1.smx | 37968 | 8 | 16 | 357da8daf37a76f39eb5eaa5a073eeeb482b8910935af0fe5063af57911c7e7f | True |
| n_tree_wax_palm_x2.smx | 135256 | 8 | 16 | d1967e843211374eaf6784a850ef5731e78c25f681d4a1eaca4a3785e3123010 | True |
| n_tree_willow_x1.smx | 14258 | 3 | 6 | 2f56bb1fcf2c792f96b66701c80ed2097127792fc39bacb1c69a6e61b422c7b8 | True |
| n_tree_willow_x2.smx | 50741 | 3 | 6 | a4f2d886c15b5ca895fb9e80001a397c7187351e4f209f7a99c7f251ac01d40a | True |

## 有意留空的树桩与元数据差异

用户确认：树桩留空是为了在游戏中屏蔽，不需要视觉分析。这里只描述精确复现必须保留的结构。stump_bamboo_x1、stump_baobab_x1、stump_generic_x1 各 1,409 字节、3 帧：0×0 轮廓声明 payload=196，实际只有一个零流长度字段（4 字节）；空主图没有 draw，却保留一块 5 字节像素数据；compressed 比实际体长小 60。它们是有意留空的占位文件，不在此修复。

其余 107 个文件满足 uncompressed=Σframe.smp_size，与 Workshop writer 额外加 4×frame_count 的约定不同。历史生成原因未确认。所有声明值都保留，不参与读游标。新生成文件采用下方明确的 Workshop 约定。未知 uint32 全为 0；memo 既有全零，也有 v1.41 SLX Studio。

In [4]:
lines = ['| File | compressed minus actual body | uncompressed minus frame sum | Stale layers | Surplus blocks |', '|---|---:|---:|---:|---:|']
for r in records:
    if r['compressed_delta'] or r['stale_layers'] or r['extra_slot_layers']:
        lines.append(f"| {r['file']} | {r['compressed_delta']} | {r['smp_delta']+4*r['frames']} | {len(r['stale_layers'])} | {len(r['extra_slot_layers'])} |")
display(Markdown('\n'.join(lines)))
assert all(r['smp_delta'] == -4*r['frames'] for r in records if not r['stale_layers'])
print('Unknown values:', summary['values']['unknown'])
print('Unused slot values:', summary['values']['padding'])


| File | compressed minus actual body | uncompressed minus frame sum | Stale layers | Surplus blocks |
|---|---:|---:|---:|---:|
| n_tree_stump_bamboo_x1.smx | -60 | 5268 | 3 | 3 |
| n_tree_stump_baobab_x1.smx | -60 | 5268 | 3 | 3 |
| n_tree_stump_generic_x1.smx | -60 | 5268 | 3 | 3 |

Unknown values: [0]
Unused slot values: [(0, 0)]


## 选择文件与查看字节账本

仅输出结构，不打印原始 bytes 或像素数组。修改文件名和帧号可探索其他文件。

In [5]:
SELECTED_FILE = 'n_tree_pine_x1.smx'
FRAME_INDEX = 0
model = parse_smx((GRAPHICS / SELECTED_FILE).read_bytes())
frame = model['frames'][FRAME_INDEX]
print({k:v for k,v in frame.items() if k != 'layers'})
for l in frame['layers']:
    print({k:l[k] for k in ('kind','offset','end','width','height','hotspot','length','unknown','used_slots','op_counts')})
print('First structural spans:', model['spans'][:8])
assert rebuild(model) == (GRAPHICS / SELECTED_FILE).read_bytes()


{'offset': 32, 'end': 4774, 'flags': 3, 'palette': 30, 'smp_size': 3840}
{'kind': 'main', 'offset': 38, 'end': 2259, 'width': 33, 'height': 74, 'hotspot': (16, 71), 'length': 2205, 'unknown': 0, 'used_slots': 1395, 'op_counts': {1: 78, 3: 74, 0: 4}}
{'kind': 'shadow', 'offset': 2259, 'end': 4774, 'width': 94, 'height': 69, 'hotspot': (47, 45), 'length': 2499, 'unknown': 0, 'used_slots': 0, 'op_counts': {1: 67, 0: 29, 3: 41}}
First structural spans: [(0, 32, 'file.header'), (32, 38, 'frame.0.header'), (38, 54, 'frame.0.main.header'), (54, 58, 'frame.0.main.edge.0'), (58, 62, 'frame.0.main.edge.1'), (62, 66, 'frame.0.main.edge.2'), (66, 70, 'frame.0.main.edge.3'), (70, 74, 'frame.0.main.edge.4')]


## 从像素生成与后续编辑

encode_rows 不读取原命令或边界，而是根据 None、普通色与玩家色重建每行最多 64 长 run，重算边界，连续打包主图，末块补零。canonical_model 重算所有实际长度，采用 Workshop 的 SMP 元数据约定：每层固定开销 32+8H，主图像素 4 字节，命令与阴影内联值另计；每帧向上对齐 64，全局另加 4×帧数。

这条路径保留所有图层像素、尺寸、锚点、flags、palette，但正常化元数据与填充，不要求与原文件字节相同。精确复现用 rebuild；编辑 rows 后生成新文件用 canonical_model 再 rebuild。尺寸改动须同步 width/height/hotspot。新纹理量化为调色板索引不在本报告实现范围。

In [6]:
def encode_rows(layer: dict, enhanced: bool) -> dict:
    # Canonical encoding from pixels, independent of original commands/edges.
    l = {k: layer[k] for k in ('kind','width','height','hotspot','unknown')}
    edges, stream, slots = [], bytearray(), []
    for row in layer['rows']:
        visible = [i for i,p in enumerate(row) if p is not None]
        if not visible:
            edges.append((65535,65535))
            continue
        left,right=visible[0],visible[-1]+1
        edges.append((left,l['width']-right))
        x=left
        while x<right:
            op=0 if row[x] is None else row[x][0]
            end=x+1
            while end<min(right,x+64) and (0 if row[end] is None else row[end][0])==op:
                end+=1
            stream.append(((end-x-1)<<2)|op)
            if op:
                if l['kind']=='main':slots.extend((p[1],p[2]) for p in row[x:end])
                elif l['kind']=='shadow':stream.extend(p[1] for p in row[x:end])
            x=end
        stream.append(3)
    group=2 if enhanced else 4
    used=len(slots)
    slots.extend([(0,0)]*((-used)%group))
    data=pack_pixels(slots,enhanced) if l['kind']=='main' else b''
    l.update(edges=edges,commands=bytes(stream),data=data,
             length=4*l['height']+(8 if l['kind']=='main' else 4)+len(stream)+len(data))
    decode_layer(l,enhanced)
    return l

def canonical_model(model: dict) -> dict:
    # SMX-Workshop size convention, not the inconsistent source convention.
    result={k:model[k] for k in ('version','comment')}
    frames=[]
    for f in model['frames']:
        layers=[encode_rows(l,bool(f['flags'] & 8)) for l in f['layers']]
        size=0
        for l in layers:
            size+=32+8*l['height']
            size+=len(l['commands'])+(4*l['used_slots'] if l['kind']=='main' else 0)
        frames.append(dict(flags=f['flags'],palette=f['palette'],smp_size=(size+63)&~63,layers=layers))
    result.update(frames=frames,uncompressed=4*len(frames)+sum(f['smp_size'] for f in frames))
    result['compressed']=sum(6+sum(16+l['length'] for l in f['layers']) for f in frames)
    return result

def verify_generation(directory: Path) -> int:
    count=0
    for path in sorted(directory.iterdir()):
        original=parse_smx(path.read_bytes())
        generated=parse_smx(rebuild(canonical_model(original)))
        for a,b in zip(original['frames'],generated['frames'],strict=True):
            assert (a['flags'],a['palette'])==(b['flags'],b['palette'])
            for x,y in zip(a['layers'],b['layers'],strict=True):
                assert (x['kind'],x['hotspot'],x['rows'])==(y['kind'],y['hotspot'],y['rows'])
        count+=1
    return count


In [7]:
assert verify_generation(GRAPHICS) == len(records)
print('Canonical regeneration: all', len(records), 'files preserve every pixel and hotspot.')
import copy
edited = copy.deepcopy(model)
main = next(l for l in edited['frames'][FRAME_INDEX]['layers'] if l['kind'] == 'main')
y,x = next((y,x) for y,row in enumerate(main['rows']) for x,p in enumerate(row) if p is not None)
op,index,smudge = main['rows'][y][x]
main['rows'][y][x] = (op,(index+1)%1024,smudge)
new_bytes = rebuild(canonical_model(edited))
new_model = parse_smx(new_bytes)
new_main = next(l for l in new_model['frames'][FRAME_INDEX]['layers'] if l['kind'] == 'main')
assert new_main['rows'] == main['rows']
print('Pixel edit and reparse passed in memory; reference untouched.')


Canonical regeneration: all 110 files preserve every pixel and hotspot.
Pixel edit and reparse passed in memory; reference untouched.


## 未出现分支的合成样本

检验玩家色、阴影 op=2、非空轮廓、全空行、透明洞、64 长 run、负锚点与两种压缩。这只验证实现，不宣称样本或游戏已覆盖这些分支。

In [8]:
import random
rng = random.Random(22014)
for enhanced in (False, True):
    slots = [(rng.randrange(1024), rng.randrange(1024) if enhanced else 0) for _ in range(40)]
    assert unpack_pixels(pack_pixels(slots,enhanced),enhanced) == slots
    layers=[]
    for kind in ('main','shadow','outline'):
        row=[]
        for x in range(130):
            if x==65: row.append(None)
            elif kind=='main': row.append((1 if x<70 else 2,(x*37)%1024,(x*19)%1024 if enhanced else 0))
            elif kind=='shadow': row.append((1 if x<70 else 2,x%256))
            else: row.append((1 if x<70 else 2,))
        layer=dict(kind=kind,width=130,height=2,hotspot=(-3,1),unknown=0,rows=[row,[None]*130])
        encoded=encode_rows(layer,enhanced)
        assert encoded['rows']==layer['rows']
        layers.append(encoded)
    fixture=dict(version=2,comment=b'\0'*16,frames=[dict(flags=7|(8 if enhanced else 0),palette=30,smp_size=0,layers=layers)])
    data=rebuild(canonical_model(fixture))
    assert rebuild(parse_smx(data))==data
print('Synthetic branch checks passed.')


Synthetic branch checks passed.

## 调色板与锚点对齐预览

palette ID 20、28、30 对应本地配置 b_east.pal、b_scen.pal、n_trees.pal。SMX 不包含色表，不能仅靠它恢复 RGBA。读取 JASC-PAL 时保留第四列 alpha，不无条件当成 RGB。不同层的像素 (x,y) 映射到共同原点 (x−hotspot_x,y−hotspot_y)。

单元格在内存中分别展示主图、阴影、轮廓与叠加。树桩无需预览。玩家色预览需另外指定玩家色表，此样本无玩家色。准确游戏渲染未验证。此单元格标记 transient_asset_output，交付版清除其输出，避免把派生素材纳入 git。

In [9]:
import numpy as np
import matplotlib.pyplot as plt

def load_palette(palette_id: int) -> np.ndarray:
    mapping={}
    for line in (PALETTE_ROOT/'palettes.conf').read_text().splitlines():
        line=line.strip()
        if line and not line.startswith('//'):
            number,name=line.split(',',1)
            mapping[int(number)]=name.strip()
    lines=(PALETTE_ROOT/mapping[palette_id]).read_text().splitlines()
    assert lines[0]=='JASC-PAL'
    count=int(lines[2])
    colors=[]
    for line in lines[3:3+count]:
        color=[int(v) for v in line.split()]
        colors.append(color if len(color)==4 else color+[255])
    assert len(colors)==count
    return np.asarray(colors,dtype=np.uint8)

palette=load_palette(frame['palette'])
active=[l for l in frame['layers'] if l['width'] and l['height']]
x0=min(-l['hotspot'][0] for l in active); y0=min(-l['hotspot'][1] for l in active)
x1=max(l['width']-l['hotspot'][0] for l in active); y1=max(l['height']-l['hotspot'][1] for l in active)
canvases={kind:np.zeros((y1-y0,x1-x0,4),dtype=np.uint8) for kind in ('main','shadow','outline')}
for l in active:
    canvas=canvases[l['kind']]
    for y,row in enumerate(l['rows']):
        for x,p in enumerate(row):
            if p is None: continue
            if l['kind']=='main':
                assert p[0]==1, 'Player-color preview requires a player palette.'
                color=palette[p[1]]
            elif l['kind']=='shadow': color=(0,0,0,p[1])
            else: color=(255,0,255,255)
            canvas[y-l['hotspot'][1]-y0,x-l['hotspot'][0]-x0]=color
fig,axes=plt.subplots(1,4,figsize=(14,5))
for ax,kind in zip(axes[:3],('main','shadow','outline')):
    ax.imshow(canvases[kind]); ax.set_title(kind)
axes[3].imshow(canvases['shadow']); axes[3].imshow(canvases['main']); axes[3].imshow(canvases['outline'])
axes[3].set_title('Aligned overlay')
for ax in axes:
    ax.set_facecolor('#dddddd'); ax.axis('off')
fig.suptitle(f'{SELECTED_FILE} / frame {FRAME_INDEX}')
plt.tight_layout(); plt.show()


## 后续 agent 操作要点

1. 调整开头 MOD_ROOT，运行全部单元格。核心解析仅用标准库，预览用项目现有 NumPy/Matplotlib。测试环境 Python 3.14.4。
2. 原字节复现用 rebuild(parse_smx(raw))；编辑生成用 rebuild(canonical_model(edited))。读写成功标准分别为字节一致与像素/尺寸/锚点一致。
3. 主图保留像素类别与 10-bit 索引；有 smudge 时保留全部 10 位。阴影值 0 与透明 None 区分。
4. 不按声明层长跳转。空行不消费 EOL。位掩码用 1/2/4/8；打包槽索引跨行连续。
5. 树桩有意屏蔽，保留占位素材，不补画、不自动修复。修改树木尺寸时同步锚点并检查与阴影对齐。
6. 导出到 tmp 或新 mod 目录，再解析检查。reference 只读，游戏内兼容性单独验证。
7. 提交前清除图像、原始 bytes、像素数组输出；不添加复制的 SMX、palette 或 PNG。本仓库 tmp 已加入 gitignore。